# Whole images or patches: a cross-validation on Kaggle

Compares the two consistent ways of training and predicting, on all five folds of
the patient-level splits, on a free Kaggle GPU. Research use only: this is not a
medical device.

- **`whole`**: trained on whole images, which are predicted in one pass.
- **`patches`**: trained on 512 px patches (half of those of images with a mass
  placed around the mass), and whole images are predicted in sliding windows of the
  patch size.

Both run 60 epochs of 64 optimizer steps, with the same splits, augmentation, loss
and optimizer: whole images in batches of 4 (as many as fit a T4 at 1024 x 800),
patches in batches of 8 (about 3.3 and 2.1 million pixels per step). The notebook
rebuilds the data from the INbreast dataset, trains every fold of both, scores the
last epoch of each fold with `mammo-seg evaluate`, and compares the folds: the mean
and the standard deviation over the five folds, and the difference per fold, which
compares the two on the same validation images.

**Once per notebook (settings panel):**
- *Add input*: the INbreast dataset.
- Accelerator: *GPU T4* (or P100). Choose it before running anything: switching it
  later restarts the machine and removes the installed packages.
- Internet: *on* (the repository and the pretrained encoder weights are downloaded).

**Run** the cells in order; only the first one needs editing. Never put `kaggle.json`
or any other credential in a notebook.

**Time** (an estimate from earlier runs, not measured for this notebook): about 25
to 35 minutes per fold and design on a T4, so the ten runs take about 5 to 6 hours,
within one session. Training stops cleanly before `MAX_MINUTES`
and saves `last.pt` after every epoch. If a session ends first, save a version of
the notebook (its output holds the `runs` folder), add that output as an input, set
`PREVIOUS_RUNS` to its `runs` folder and run the cells again: an unfinished run is
resumed and a finished one returns at once. The folds run in turn for both designs,
so the folds that are done always form pairs.

In [ ]:
from pathlib import Path

FOLDS = [0, 1, 2, 3, 4]
EPOCHS = 60
MAX_MINUTES = 11 * 60  # for all runs together; training stops cleanly before this
# The options of `mammo-seg train` of each design; the inference follows the crops.
EXPERIMENTS = {
    "whole": "--batch-size 4 --samples-per-epoch 256 --patience 0",
    "patches": "--crop-size 512 --batch-size 8 --samples-per-epoch 512 --patience 0",
}
REPO_URL = "https://github.com/GiannisPapa97/mammo_seg.git"
REPO_REF = "main"  # branch or tag to install
PREVIOUS_RUNS = None  # for example Path("/kaggle/input/my-notebook/runs")

## 1. Setup

Installs the package (`.[cloud]` leaves the PyTorch of the notebook, a GPU build,
untouched), finds the dataset and rebuilds the derived data: manifest, patient-level
splits (stratified on masses, mass size and breast density), masks and preprocessed
images (a few minutes). A step whose output exists is skipped. The repository is
cloned to `repo`, not `mammo_seg`: a folder of that name next to the notebook would
hide the installed package from Python.

The cells below only run `mammo-seg`, or read its results with `json` and `pandas`.
A cell of your own that imports `mammo_seg` or `scipy` in this session fails with a
numpy error, because the install upgraded numpy after the notebook had loaded it:
restart the session once (*Run* → *Restart session*; files and installed packages
are kept), run the first cell and this one again, and then use your cell.

In [ ]:
WORK = Path("/kaggle/working")
REPO = WORK / "repo"
PROCESSED = WORK / "data" / "processed"
RUNS = WORK / "runs"
MANIFEST = PROCESSED / "manifest.json"
SPLITS = PROCESSED / "splits.json"
MASKS = PROCESSED / "masks"
PREPROCESSED = PROCESSED / "preprocessed"

found = sorted(Path("/kaggle/input").rglob("INbreast.csv"))
assert found, "attach the INbreast dataset as an input of the notebook"
DATASET_DIR = found[0].parent
print("dataset:", DATASET_DIR)

!test -d {REPO} || git clone -q --branch {REPO_REF} {REPO_URL} {REPO}
!pip install -q "{REPO}[cloud]"
!nvidia-smi -L

# The code of every session: the time and the commit, kept with the runs.
RUNS.mkdir(parents=True, exist_ok=True)
VERSION = RUNS / "code_version.txt"
previous_version = Path(PREVIOUS_RUNS) / "code_version.txt" if PREVIOUS_RUNS else None
if previous_version and previous_version.exists() and not VERSION.exists():
    VERSION.write_text(previous_version.read_text())
!echo "$(date -u +%FT%TZ) $(git -C {REPO} rev-parse HEAD)" | tee -a {VERSION}

PROCESSED.mkdir(parents=True, exist_ok=True)
if not MANIFEST.exists():
    !mammo-seg manifest --dataset-dir "{DATASET_DIR}" --output {MANIFEST}
if not SPLITS.exists():
    !mammo-seg splits --folds 5 --test-fraction 0.2 \
        --manifest {MANIFEST} --output {SPLITS}
if not MASKS.exists():
    !mammo-seg masks --manifest {MANIFEST} --output-dir {MASKS}
if not (PREPROCESSED / "preprocessing.json").exists():
    !mammo-seg preprocess --manifest {MANIFEST} --dataset-dir "{DATASET_DIR}" \
        --masks-dir {MASKS} --output-dir {PREPROCESSED}

## 2. Train

One run per fold and design, in `runs/<design>/fold_<k>`: `last.pt`, `best.pt` and
`history.jsonl`. The folds run in turn for both designs. `MAX_MINUTES` is shared:
each run gets what is left. `--device cuda` makes a notebook without a GPU an error
instead of a slow run on the CPU.

In [ ]:
import shutil
import time

started = time.monotonic()
for fold in FOLDS:
    for name, options in EXPERIMENTS.items():
        minutes = int(MAX_MINUTES - (time.monotonic() - started) / 60)
        if minutes < 1:
            print(f"{name}, fold {fold}: MAX_MINUTES is used up; continue later")
            continue
        runs = RUNS / name
        fold_dir = runs / f"fold_{fold}"
        previous = (
            Path(PREVIOUS_RUNS) / name / f"fold_{fold}" if PREVIOUS_RUNS else None
        )
        if previous and previous.exists() and not fold_dir.exists():
            shutil.copytree(previous, fold_dir)
        resume = "--resume" if (fold_dir / "last.pt").exists() else ""
        state = "resuming" if resume else "new run"
        print(f"=== {name}, fold {fold}: {state}, {minutes} min left; {options}")
        !mammo-seg train --manifest {MANIFEST} --splits {SPLITS} \
            --preprocessed-dir {PREPROCESSED} --output-dir {runs} --device cuda \
            --num-workers 2 --epochs {EPOCHS} {options} \
            --fold {fold} --max-minutes {minutes} {resume}

## 3. Evaluate

Scores the last epoch of every fold of a design on its own validation fold, once all
its folds have finished (each model predicts as it was validated in training: whole
images in one pass, patches in sliding windows). The table it prints pools the folds
over a grid of thresholds and minimum region sizes, with the masses found, the false
regions per image and the subgroups; everything is written to
`runs/<design>/evaluation_validation.json`, which the next cell reads.

In [ ]:
def finished(name: str) -> bool:
    histories = [RUNS / name / f"fold_{fold}" / "history.jsonl" for fold in FOLDS]
    return all(
        path.exists() and len(path.read_text().splitlines()) >= EPOCHS
        for path in histories
    )


folds = " ".join(str(fold) for fold in FOLDS)
for name in EXPERIMENTS:
    if not finished(name):
        print(f"{name}: not every fold has run its {EPOCHS} epochs; train first")
        continue
    runs = RUNS / name
    output = runs / "evaluation_validation.json"
    print(f"=== {name}")
    !mammo-seg evaluate --manifest {MANIFEST} --splits {SPLITS} \
        --preprocessed-dir {PREPROCESSED} --runs-dir {runs} --output {output} \
        --device cuda --num-workers 2 --folds {folds}

## 4. Compare

Every fold at the same setting, a threshold of 0.5 and no minimum region size: the
models as they are, without post-processing chosen on the validation folds. Then the
mean and the standard deviation over the folds, and the difference per fold
(patches minus whole images). The folds share most of their training patients, so
their standard deviation understates the uncertainty; a difference smaller than the
spread of the per-fold differences, or not of the same sign in most folds, is not
convincing. Last, the best setting of each design over the pooled folds (by the
balanced Dice), which is slightly optimistic because it is chosen on these images.

In [ ]:
import json
import math

import pandas as pd

SETTING = {"threshold": 0.5, "min_area_mm2": 0.0}
METRICS = {
    "Dice, with a mass": ("summary", "with_mass", "dice"),
    "sensitivity": ("summary", "with_mass", "sensitivity"),
    "precision": ("summary", "with_mass", "precision"),
    "false alarms": ("summary", "without_mass", "false_alarm_rate"),
    "masses found": ("found_masses",),
    "false regions per image": ("false_regions_per_image",),
    "Dice, all images": ("summary", "all_images", "dice"),
    "balanced Dice": ("balanced_dice",),
}


def value(result: dict, path: tuple[str, ...]) -> float:
    for key in path:
        result = result[key]
    return math.nan if result is None else float(result)


evaluations, per_fold = {}, {}
for name in EXPERIMENTS:
    path = RUNS / name / "evaluation_validation.json"
    if not path.exists():
        print(f"{name}: not evaluated yet")
        continue
    evaluations[name] = evaluation = json.loads(path.read_text())
    rows = {}
    for fold, results in evaluation["per_fold"].items():
        chosen = next(r for r in results["settings"] if r["setting"] == SETTING)
        rows[int(fold)] = {
            label: value(chosen, keys) for label, keys in METRICS.items()
        }
    per_fold[name] = pd.DataFrame(rows).T.sort_index()

for name, frame in per_fold.items():
    print(f"{name}: every fold at a threshold of 0.5, no minimum region size")
    display(frame.round(3))

if per_fold:
    print("mean ± standard deviation over the folds")
    display(
        pd.DataFrame(
            {
                name: frame.mean().map("{:.3f}".format)
                + " ± "
                + frame.std().map("{:.3f}".format)
                for name, frame in per_fold.items()
            }
        )
    )

if {"whole", "patches"} <= per_fold.keys():
    difference = per_fold["patches"] - per_fold["whole"]
    print("patches minus whole images, per fold (the same validation images)")
    display(difference.round(3))
    display(
        pd.DataFrame(
            {
                "mean": difference.mean(),
                "standard deviation": difference.std(),
                "folds with patches higher": (difference > 0).sum(),
            }
        ).round(3)
    )

for name, evaluation in evaluations.items():
    settings = [
        r for r in evaluation["pooled"]["settings"] if r["balanced_dice"] is not None
    ]
    best = max(settings, key=lambda r: r["balanced_dice"])
    scores = {label: round(value(best, keys), 3) for label, keys in METRICS.items()}
    print(f"{name}: best pooled setting {best['setting']}: {scores}")